<a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/agents/gemini_data_analytics/a2a_http_sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini Data Analytics: A2A HTTP API Sample

This notebook demonstrates how to interact with the **DataA2AService** using standard HTTP requests. This is useful for environments where a high-level SDK is not available or when you want to minimize dependencies.

# Background and Overview
The **Conversational Analytics API** (also known as Gemini Data Analytics) lets you chat with your BigQuery or Looker data anywhere. This notebook demonstrates how to use the **A2A (Agent-to-Agent)** interface via standard HTTP requests. This is useful for environments where a high-level SDK is not available or when you want to minimize dependencies.

This is a **Pre-GA** product. See [documentation](https://cloud.google.com/gemini/docs/conversational-analytics-api/overview) for more details.

Please provide feedback to conversational-analytics-api-feedback@google.com
<br>
### This notebook will help you
1. Authenticate to Google Cloud
2. Retrieve the Agent Card
3. Send asynchronous messages and poll for results
4. Process agent outputs (Artifacts)
5. Cancel active tasks


In [ ]:
# @title Setup and Authentication
import json
import time
import uuid
from google.auth import default
from google.auth.transport.requests import Request
from google.colab import auth
import requests

# Authenticate the user
auth.authenticate_user()

# Get credentials and project ID
creds, project_id = default()
creds.refresh(Request())
access_token = creds.token

ENDPOINT = "https://geminidataanalytics.googleapis.com"
LOCATION = "global"  # @param {type:"string"}
AGENT_ID = "bigquery-ca"  # @param {type:"string"}

TENANT = f"projects/{project_id}/locations/{LOCATION}/dataAgents/{AGENT_ID}"
HEADERS = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json",
}
print(f"Target Tenant: {TENANT}")

## 1. Get Agent Card

First, let's retrieve the Agent Card to verify connectivity and see what the agent can do.

In [ ]:
url = f"{ENDPOINT}/v1beta/card"
params = {"tenant": TENANT}

try:
  response = requests.get(url, headers=HEADERS, params=params, timeout=30)
  response.raise_for_status()
  print("Agent Card:")
  print(json.dumps(response.json(), indent=2))
except Exception as e:
  print(f"Error fetching agent card: {e}")

## 2. Send Message (Asynchronous + Polling)

For long-running tasks, use `blocking=False`. This returns a `Task` object immediately, which you can poll for status.

In [ ]:
def send_async_message(query):
  url = f"{ENDPOINT}/v1beta/message:send"
  payload = {
      "tenant": TENANT,
      "message": {
          "message_id": f"msg-{uuid.uuid4()}",
          "context_id": "sample-context",
          "role": "ROLE_USER",
          "content": [{"text": query}],
      },
      "configuration": {"blocking": False},
  }

  response = requests.post(url, headers=HEADERS, json=payload, timeout=30)
  response.raise_for_status()

  task = response.json().get("task")
  print(f"Task created: {task.get('id')}")
  return task.get("id")


def poll_task(task_id, max_retries=15):
  url = f"{ENDPOINT}/v1beta/tasks/{task_id}"
  retry_count = 0
  wait_time = 2

  while retry_count < max_retries:
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()

    task = response.json()
    state = task.get("status", {}).get("state")
    print(f"Current State: {state}")

    if state in [
        "TASK_STATE_COMPLETED",
        "TASK_STATE_FAILED",
        "TASK_STATE_CANCELLED",
    ]:
      return task

    time.sleep(wait_time)
    # Simple exponential backoff capped at 10s
    wait_time = min(wait_time * 1.5, 10)
    retry_count += 1

  print("Polling timed out.")
  return None


try:
  task_id = send_async_message("Generate a summary of sales trends")
  if task_id:
    final_task = poll_task(task_id)
    if final_task:
      print("Final Task Result:")
      print(json.dumps(final_task, indent=2))
except Exception as e:
  print(f"Error during messaging/polling: {e}")

## 3. Processing Agent Outputs (Artifacts)

Agents often produce **Artifacts** (structured data, files, or references). Here is how to parse them from a completed task.

In [ ]:
def process_artifacts(task):
  artifacts = task.get("artifacts", [])
  if not artifacts:
    print("No artifacts found.")
    return

  print(f"Found {len(artifacts)} artifacts:")
  for art in artifacts:
    name = art.get("name", "Unnamed")
    art_type = (
        art.get("metadata", {})
        .get("fields", {})
        .get("type", {})
        .get("stringValue", "Unknown")
    )
    print(f"- [{art_type}] {name}: {art.get('description')}")

    # Example: If it is a GCS path
    if "gcs_path" in art.get("metadata", {}).get("fields", {}):
      path = art["metadata"]["fields"]["gcs_path"]["stringValue"]
      print(
          "  GCS Link:"
          f" https://console.cloud.google.com/storage/browser/{path.replace('gs://', '')}"
      )


# Use the final_task from the previous step
if "final_task" in locals():
  process_artifacts(final_task)

## 4. Cancel an Active Task

If a task is taking too long or was sent in error, you can cancel it.

In [ ]:
def cancel_task(task_id):
  url = f"{ENDPOINT}/v1beta/tasks/{task_id}:cancel"
  response = requests.post(url, headers=HEADERS, timeout=30)
  response.raise_for_status()

  print(f"Task {task_id} cancellation requested.")
  return response.json()


# Example usage (start a task and cancel it immediately)
try:
  new_task_id = send_async_message("A long running query")
  if new_task_id:
    cancel_task(new_task_id)
except Exception as e:
  print(f"Error during cancellation demo: {e}")

## 5. Cleanup

It is good practice to clean up any temporary resources or state created during your session.

In [ ]:
# @title Resource Cleanup
print(
    "No specific cloud resources were created in this demo that require manual"
    " deletion (e.g., storage buckets)."
)
print(
    "However, you can use this section to reset any local session state if"
    " needed."
)